# LowBitSparse Colab 验证手册

用于在 Colab A100 上复现 M0/M1: FP16 基线、RTN/GPTQ/AWQ 权重量化、WikiText-2 PPL、理论压缩比和汇总表。

## 推荐执行顺序

1. 环境检查和依赖安装。
2. 本地逻辑验证: `pytest` + `cpu_smoke.py`。
3. 快速单点验证: RTN INT4 只评少量样本。
4. M1 核心验收: baseline + INT8 + INT4 + GPTQ + AWQ。
5. 可选完整消融: `scripts/run_sweep.py`。

In [ ]:
# 1) 确认 GPU。M1 正式评测建议使用 A100。
!nvidia-smi -L

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [9]:
# 2) 挂载 Google Drive。结果写在 Drive 项目目录下,可避免 Colab 断连丢失。
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# 3) 获取代码。若 Drive 中已存在仓库,直接进入;否则 clone 一份。
import os
import subprocess

REPO_URL = "https://github.com/honey-floria/LowBitSparse.git"
PROJECT_DIR = "/content/drive/MyDrive/LowBitSparse"

if not os.path.exists(os.path.join(PROJECT_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())
!git rev-parse --short HEAD
!git status --short
!ls

cwd: /content/drive/MyDrive/LowBitSparse
f05613f
Refresh index: 100% (48/48), done.
configs  lowbitsparse  main.py	  requirements.txt  scripts
doc	 LowBitSparse  README.MD  results	    tests


如果你确认 Drive 中没有未提交改动,可以手动执行下一格更新代码。若 `git status --short` 有本地改动,先不要 pull。

In [ ]:
# 可选:更新仓库。若有本地改动导致失败,先保留现场再处理。
# !git pull --ff-only

In [11]:
# 4) 安装依赖。Colab 自带 torch,requirements 只做宽松约束。
%pip install -q -r requirements.txt

import torch, transformers, datasets, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.11.0+cu128
transformers: 5.13.1
datasets: 4.0.0
accelerate: 1.14.0


In [12]:
# 5) 可选:从 Colab Secrets 读取 Hugging Face token。
# 公开模型通常不需要 token;若遇到下载限流或私有模型,在 Secrets 中添加 HF_TOKEN。
import os
from google.colab import userdata

try:
    token = userdata.get("HF_TOKE")
except Exception:
    token = None

if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN 已设置")
else:
    print("未设置 HF_TOKEN,将按公开下载流程执行")

HF_TOKEN 已设置


## 本地逻辑验证

这一步不下载 Qwen,用于确认量化数学、校准 hook、Linear 替换和压缩统计没有被改坏。

In [ ]:
!python -m pytest -q
!python scripts/cpu_smoke.py

## 快速单点验证

只跑 RTN INT4,并把 PPL 评测限制到 4 个窗口。适合验证 Colab 环境、模型下载、量化和结果落盘。

In [ ]:
# 生成一个临时 smoke config,不修改正式配置文件。
import copy
import yaml
from pathlib import Path

base = yaml.safe_load(Path("configs/qwen0.5b_int4.yaml").read_text(encoding="utf-8"))
smoke = copy.deepcopy(base)
smoke["exp_id"] = "m1_rtn_int4_g128_smoke"
smoke["eval"]["max_samples"] = 4
Path("configs/_colab_smoke_int4.yaml").write_text(
    yaml.safe_dump(smoke, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)
print(Path("configs/_colab_smoke_int4.yaml").read_text(encoding="utf-8"))

In [ ]:
!python main.py quant --config configs/_colab_smoke_int4.yaml

## M1 核心验收

这组命令生成 M1 最关键的对比: FP16 baseline、RTN INT8、RTN INT4、GPTQ INT4、AWQ INT4。

In [13]:
!python main.py eval  --config configs/qwen0.5b_base.yaml
!python main.py quant --config configs/qwen0.5b_int8.yaml
!python main.py quant --config configs/qwen0.5b_int4.yaml
!python main.py quant --config configs/qwen0.5b_gptq_int4.yaml
!python main.py quant --config configs/qwen0.5b_awq_int4.yaml

!python scripts/summarize.py --results results --out results/m1_summary.md

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 1920.66it/s]
[01:55:26] INFO lowbitsparse: 模型已加载: Qwen/Qwen2.5-0.5B-Instruct
[01:55:26] INFO lowbitsparse: 参数量 494.033M, 体积 942.3MB
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
ppl:  99% 146/147 [00:05<00:00, 25.07it/s]
[01:55:35] INFO lowbitsparse: WikiText-2 PPL = 14.2445
[01:55:58] INFO lowbitsparse: prefill 28.9ms, decode 38.2 tok/s, peak 4574.341MB
[01:55:58] INFO lowbitsparse: 结果已保存: results/m0_fp16_baseline.json
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 2001.72it/s]
[01:56:09] INFO lowbitsparse: 模型已加载: Qwen/Qwen2.5-0.5B-Instruct
[01:56:09] INFO lowbitsparse: 已量化 168 个 Linear (method=rtn, bits=8, group=128, sym=False)
[01:56:09] INFO lowbitsparse: 量化后 

In [ ]:
# 查看汇总表。重点看 PPL、Delta PPL、压缩比和等效 bit。
from pathlib import Path
print(Path("results/m1_summary.md").read_text(encoding="utf-8"))

## 可选:完整 M1 消融

`run_sweep.py` 会扫描 method × bit × group_size。耗时明显长于核心验收,适合最终补表。先用 `--smoke` 确认流程,再打开完整 sweep。

In [ ]:
# 可选:全组合冒烟。每组只评 4 个 PPL 窗口,但 GPTQ/AWQ 仍会做校准。
# !python scripts/run_sweep.py --smoke
# !python scripts/summarize.py --results results --out results/m1_summary.md

In [ ]:
# 可选:完整消融。确认时间和额度足够后再执行。
RUN_FULL_SWEEP = False

if RUN_FULL_SWEEP:
    !python scripts/run_sweep.py
    !python scripts/summarize.py --results results --out results/m1_summary.md
else:
    print("RUN_FULL_SWEEP=False,已跳过完整消融")

## 结果备份

如果项目目录已经在 Drive 中,结果天然保存在 Drive。下面这格会额外复制一份带时间戳的 results 快照,避免覆盖历史实验。

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

src = Path("results").resolve()
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dst = Path("/content/drive/MyDrive/LowBitSparse_results") / stamp
dst.parent.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copytree(src, dst)
    print("results 快照已保存到:", dst)
else:
    print("results 目录不存在,没有可备份内容")